## Embdeddings and Vectorstores 

In [38]:
import os
import openai
import sys
sys.path.append('../..')

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

openai.api_key  = os.environ['OPENAI_API_KEY']

In [39]:
from langchain_community.document_loaders import PyPDFLoader

# Load PDF
loaders = [
    # Duplicate documents on purpose - messy data
    PyPDFLoader("docs/MachineLearning-Lecture01.pdf"),
    PyPDFLoader("docs/MachineLearning-Lecture01.pdf"),
    PyPDFLoader("docs/MachineLearning-Lecture02.pdf"),
    PyPDFLoader("docs/MachineLearning-Lecture03.pdf")
]
docs = []
for loader in loaders:
    docs.extend(loader.load())

In [40]:
# Split
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 150
)

In [41]:
splits = text_splitter.split_documents(docs)

In [42]:
len(splits)

208

## Embeddings

In [43]:
# !pip install langchain_openai

In [44]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings()

In [45]:
sentence1 = "i like dogs"
sentence2 = "i like canines"
sentence3 = "the weather is ugly outside"

In [46]:
embedding1 = embedding.embed_query(sentence1)
embedding2 = embedding.embed_query(sentence2)
embedding3 = embedding.embed_query(sentence3)

In [47]:
len(embedding1)

1536

In [49]:
embedding1[:4]

[-0.02755114622414112,
 -0.005382069852203131,
 -0.02582130767405033,
 -0.03305632621049881]

In [50]:
import numpy as np

np.dot(embedding1, embedding2)

0.9631511809630344

In [51]:
np.dot(embedding1, embedding3)

0.7703084186886011

In [52]:
np.dot(embedding2, embedding3)

0.7591611894209248

In [53]:
# ! pip install chromadb

In [54]:
from langchain_community.vectorstores import Chroma

In [55]:
rm -rf docs/chroma/

In [56]:
persist_directory = 'docs/chroma/'

In [57]:
import os
os.makedirs(persist_directory, exist_ok=True)

In [58]:
# !pip uninstall chromadb -y
# !pip install chromadb==0.4.24

In [59]:
import sys
print(sys.executable)

/Users/koushalsmodi/Desktop/MachineLearning/MachineLearningProjects/NLP/.venv/bin/python


In [60]:
vectordb = Chroma.from_documents(
    documents=splits,
    embedding=embedding,
    persist_directory=persist_directory
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


OperationalError: attempt to write a readonly database

In [ ]:
print(vectordb._collection.count())
# number of vectors stored in your vector database

208


## Similarity Search

In [ ]:
question = "is there an email i can ask for help"

In [ ]:
docs = vectordb.similarity_search(question,k=3)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [61]:
len(docs)

78

In [62]:
docs[0].page_content

'MachineLearning-Lecture01  \nInstructor (Andrew Ng): Okay. Good morning. Welcome to CS229, the machine \nlearning class. So what I wanna do today is just spend a little time going over the logistics \nof the class, and then we\'ll start to talk a bit about machine learning.  \nBy way of introduction, my name\'s Andrew Ng and I\'ll be instructor for this class. And so \nI personally work in machine learning, and I\'ve worked on it for about 15 years now, and \nI actually think that machine learning is the most exciting field of all the computer \nsciences. So I\'m actually always excited about teaching this class. Sometimes I actually \nthink that machine learning is not only the most exciting thing in computer science, but \nthe most exciting thing in all of human endeavor, so maybe a little bias there.  \nI also want to introduce the TAs, who are all graduate students doing research in or \nrelated to the machine learning and all aspects of machine learning. Paul Baumstarck \nworks i

In [63]:
vectordb.persist()

## Failure modes

In [64]:
question = "what did they say about matlab?"

In [65]:
docs = vectordb.similarity_search(question,k=5)

In [66]:
# Notice that we're getting duplicate chunks (because of the duplicate MachineLearning-Lecture01.pdf in the index).
# Semantic search fetches all similar documents, but does not enforce diversity docs[0] and docs[1] are indentical.

In [67]:
docs[0].page_content

'those homeworks will be done in either MATLAB or in Octave, which is sort of — I \nknow some people call it a free version of MATLAB, which it sort of is, sort of isn\'t.  \nSo I guess for those of you that haven\'t seen MATLAB before, and I know most of you \nhave, MATLAB is I guess part of the programming language that makes it very easy to \nwrite codes using matrices, to write code for numerical routines, to move data around, to \nplot data. And it\'s sort of an extremely easy to learn tool to use for implementing a lot of \nlearning algorithms.  \nAnd in case some of you want to work on your own home computer or something if you \ndon\'t have a MATLAB license, for the purposes of this class, there\'s also — [inaudible] \nwrite that down [inaudible] MATLAB — there\'s also a software package called Octave \nthat you can download for free off the Internet. And it has somewhat fewer features than \nMATLAB, but it\'s free, and for the purposes of this class, it will work for just abou

In [68]:
docs[1].page_content

'those homeworks will be done in either MATLAB or in Octave, which is sort of — I \nknow some people call it a free version of MATLAB, which it sort of is, sort of isn\'t.  \nSo I guess for those of you that haven\'t seen MATLAB before, and I know most of you \nhave, MATLAB is I guess part of the programming language that makes it very easy to \nwrite codes using matrices, to write code for numerical routines, to move data around, to \nplot data. And it\'s sort of an extremely easy to learn tool to use for implementing a lot of \nlearning algorithms.  \nAnd in case some of you want to work on your own home computer or something if you \ndon\'t have a MATLAB license, for the purposes of this class, there\'s also — [inaudible] \nwrite that down [inaudible] MATLAB — there\'s also a software package called Octave \nthat you can download for free off the Internet. And it has somewhat fewer features than \nMATLAB, but it\'s free, and for the purposes of this class, it will work for just abou

In [69]:
# We can see a new failure mode.
# The question below asks a question about the third lecture, but includes results from other lectures as well.
question = "what did they say about regression in the third lecture?"

In [70]:
docs = vectordb.similarity_search(question,k=5)

In [71]:
for doc in docs:
    print(doc.metadata)

{'author': '', 'creationdate': '2008-07-11T11:25:03-07:00', 'creator': 'PScript5.dll Version 5.2.2', 'moddate': '2008-07-11T11:25:03-07:00', 'page': 0, 'page_label': '1', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'source': 'docs/MachineLearning-Lecture03.pdf', 'title': '', 'total_pages': 16}
{'author': '', 'creationdate': '2008-07-11T11:25:05-07:00', 'creator': 'PScript5.dll Version 5.2.2', 'moddate': '2008-07-11T11:25:05-07:00', 'page': 2, 'page_label': '3', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'source': 'docs/MachineLearning-Lecture02.pdf', 'title': '', 'total_pages': 18}
{'author': '', 'creationdate': '2008-07-11T11:25:05-07:00', 'creator': 'PScript5.dll Version 5.2.2', 'moddate': '2008-07-11T11:25:05-07:00', 'page': 17, 'page_label': '18', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'source': 'docs/MachineLearning-Lecture02.pdf', 'title': '', 'total_pages': 18}
{'author': '', 'creationdate': '2008-07-11T11:25:23-07:00', 'creator': 'PScript5.dll Version 5.2.2

In [72]:
print(docs[4].page_content)

into his office and he said, "Oh, professor, professor, thank you so much for your 
machine learning class. I learned so much from it. There's this stuff that I learned in your 
class, and I now use every day. And it's helped me make lots of money, and here's a 
picture of my big house."  
So my friend was very excited. He said, "Wow. That's great. I'm glad to hear this 
machine learning stuff was actually useful. So what was it that you learned? Was it 
logistic regression? Was it the PCA? Was it the data networks? What was it that you 
learned that was so helpful?" And the student said, "Oh, it was the MATLAB."  
So for those of you that don't know MATLAB yet, I hope you do learn it. It's not hard, 
and we'll actually have a short MATLAB tutorial in one of the discussion sections for 
those of you that don't know it.  
Okay. The very last piece of logistical thing is the discussion sections. So discussion 
sections will be taught by the TAs, and attendance at discussion sections is o